# STIR-Net V1 — 28 Information-Sufficiency Causal Experiments

This notebook does **not** modify repository source files. It temporarily prototypes each suspected fix in isolation from the same trained Notebook-27 `checkpoint_best_spatial_query.pt`.

The question for every block is:

> **What decision must it make, what information does it actually receive, and does giving it better information improve the result?**

Experiments:

- **A — Center audit:** initial vs final center coverage, duplicates, missing cells.
- **B — Proposal decision logic:** independent proposal refiner vs proposal-set Transformer.
- **C — Center information:** current query-only center refinement vs direct local 3-D evidence.
- **D — Coarse masks:** 2048 nearest-label target vs occupancy-preserving targets at 2048/8192/16384 mask positions.
- **E — Mask information:** D0 only vs D0+raw vs D0+all explicit spatial evidence, plus oracle-center test.
- **F — Support geometry:** 1.0–2.5 dref support coverage around GT and predicted centers.

This is a one-scene **causal overfit experiment**, not a generalization benchmark.


In [ ]:
from __future__ import annotations

from pathlib import Path
from typing import Any
import copy, gc, json, math, shutil, time, traceback

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

from scipy.optimize import linear_sum_assignment

from learned.stirnet import StirNet
from learned.stirnet.debugging.acceptance.first_overfit import _reduced_config, _repo_root, build_real_batch
from learned.stirnet.model.matcher import build_local_support_masks, target_ids, target_masks_at_shape
from learned.stirnet.model.query_builder import QUERY_SPATIAL_PROPOSAL
from learned.stirnet.model.types import StirNetOutput, TemporalState
from learned.stirnet.training.checkpoint import load_checkpoint

try:
    from learned.stirnet.training.trainer import move_batch_to_device
except ImportError:
    from learned.stirnet.training.trainer import move_to_device as move_batch_to_device

SEED = 40266
SOURCE_ID = 9
AMP_DTYPE = torch.float16

TOKEN_CAPS = [2048, 8192, 16384]
MASK_PROBE_STEPS = 60
MASK_PROBE_LR = 2e-3

LOCAL_GRID_SIZE = 20
LOCAL_EXTENT_DREF = 1.35
LOCAL_CENTER_STEPS = 250
LOCAL_MASK_STEPS = 120
LOCAL_HEAD_LR = 2e-3

OVERCOMPLETE_NMS_DREF = 0.20
PROPOSAL_REFINER_STEPS = 300
PROPOSAL_REFINER_LR = 2e-3
PROPOSAL_MAX_MOVE_DREF = 0.60

SUPPORT_RADII_DREF = [1.0, 1.25, 1.5, 2.0, 2.5]

RUN_PROPOSAL_SET_EXPERIMENT = True
RUN_LOCAL_CENTER_EXPERIMENT = True
RUN_COARSE_MASK_PROBE = True
RUN_LOCAL_MASK_EXPERIMENT = True
RUN_SUPPORT_AUDIT = True
OPEN_NAPARI_AT_END = False

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

REPO_ROOT = _repo_root(Path.cwd())
DATA_DIR = REPO_ROOT / "data" / "learned" / "stirnet" / "first_overfit" / "BlastoSPIM1_F22_030_034"
OVERNIGHT_ROOT = REPO_ROOT / "runs" / "stirnet" / "overnight"
BEST_SPATIAL_QUERY_CHECKPOINT = None

if BEST_SPATIAL_QUERY_CHECKPOINT is None:
    runs = sorted(OVERNIGHT_ROOT.glob("27_overnight_*"), key=lambda p: p.stat().st_mtime, reverse=True)
    candidates = [r / "checkpoint_best_spatial_query.pt" for r in runs if (r / "checkpoint_best_spatial_query.pt").exists()]
    if not candidates:
        raise FileNotFoundError("No checkpoint_best_spatial_query.pt found. Set BEST_SPATIAL_QUERY_CHECKPOINT manually.")
    BEST_SPATIAL_QUERY_CHECKPOINT = candidates[0]
else:
    BEST_SPATIAL_QUERY_CHECKPOINT = Path(BEST_SPATIAL_QUERY_CHECKPOINT)

RUN_DIR = REPO_ROOT / "runs" / "stirnet" / "experiments" / f"28_information_sufficiency_{time.strftime('%Y%m%d_%H%M%S')}"
RUN_DIR.mkdir(parents=True, exist_ok=False)
LOG_PATH = RUN_DIR / "experiment.log"
RESULTS_JSONL = RUN_DIR / "results.jsonl"
ERRORS_JSONL = RUN_DIR / "errors.jsonl"

print("Repository :", REPO_ROOT)
print("Checkpoint :", BEST_SPATIAL_QUERY_CHECKPOINT)
print("Run dir    :", RUN_DIR)
print("GPU        :", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")


In [ ]:
def now_text():
    return time.strftime("%Y-%m-%d %H:%M:%S")

def _jsonable(v):
    if isinstance(v, Path):
        return str(v)
    if isinstance(v, np.generic):
        return v.item()
    if torch.is_tensor(v):
        return v.detach().cpu().item() if v.numel() == 1 else v.detach().cpu().tolist()
    if isinstance(v, float) and not math.isfinite(v):
        return None
    return v

def log(message):
    line = f"[{now_text()}] {message}"
    print(line, flush=True)
    with LOG_PATH.open("a", encoding="utf-8") as f:
        f.write(line + "\n")
        f.flush()

def append_jsonl(path, payload):
    with path.open("a", encoding="utf-8") as f:
        json.dump({k:_jsonable(v) for k,v in payload.items()}, f)
        f.write("\n")
        f.flush()

def record_result(experiment, **kwargs):
    append_jsonl(RESULTS_JSONL, {"time":now_text(), "experiment":experiment, **kwargs})

def record_error(stage, exc):
    text = "".join(traceback.format_exception(type(exc), exc, exc.__traceback__))
    log(f"ERROR in {stage}: {type(exc).__name__}: {exc}")
    append_jsonl(ERRORS_JSONL, {"time":now_text(), "stage":stage, "type":type(exc).__name__, "message":str(exc), "traceback":text})

def cleanup():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

log(f"Free disk: {shutil.disk_usage(REPO_ROOT.anchor).free / 1024**3:.2f} GiB")


In [ ]:
if device.type != "cuda":
    raise RuntimeError("Notebook 28 requires CUDA.")

batch_cpu, sample_info = build_real_batch(DATA_DIR)
b = move_batch_to_device(batch_cpu, device)
b["spatial_inputs"] = b["spatial_inputs"].to(dtype=AMP_DTYPE)
b["instance_labels"] = b["instance_labels"].to(dtype=torch.int32)

targets = batch_cpu["targets"]
target = targets[0]

cfg = _reduced_config()
cfg.proposals.enabled = True
cfg.proposals.query_mode = "spatial_proposals"

model = StirNet(cfg).to(device)
checkpoint_info = load_checkpoint(
    BEST_SPATIAL_QUERY_CHECKPOINT,
    model,
    optimizer=None,
    scheduler=None,
    scaler=None,
    map_location="cpu",
    strict=True,
    migrate_history=True,
)
model.eval()

gt_ids_all = target_ids(target).detach().cpu().long()
gt_centers_all = torch.as_tensor(target["centers_cellscale"], dtype=torch.float32)

current_labels_native = batch_cpu["instance_labels"][0].detach().cpu().numpy().astype(np.int32, copy=False)
gt_labels_native = torch.as_tensor(target["label_map"]).detach().cpu().numpy().astype(np.int32, copy=False)
spacing_native = batch_cpu["spacing_um"][0].detach().cpu().numpy().astype(np.float64)
dref_um = float(batch_cpu["dref_um"][0])

source9_gt_ids = np.unique(gt_labels_native[current_labels_native == SOURCE_ID])
source9_gt_ids = source9_gt_ids[source9_gt_ids > 0].astype(int)
source9_id_set = set(source9_gt_ids.tolist())
source9_gt_indices = torch.tensor([i for i,x in enumerate(gt_ids_all.tolist()) if int(x) in source9_id_set], dtype=torch.long)
source9_gt_centers = gt_centers_all[source9_gt_indices].float()

if len(source9_gt_ids) != 9:
    raise RuntimeError(f"Expected 9 source-9 GT cells, got {len(source9_gt_ids)}.")

log(f"Loaded step={checkpoint_info.get('step')} source9={source9_gt_ids.tolist()}")


In [ ]:
def make_empty_temporal(model, dtype):
    d_model = int(model.cfg.temporal.d_model)
    return TemporalState(
        tokens=torch.empty((0,d_model), device=device, dtype=dtype),
        ref_um=torch.empty((0,3), device=device, dtype=torch.float32),
        ref_cellscale=torch.empty((0,3), device=device, dtype=torch.float32),
        salience=torch.empty((0,1), device=device, dtype=dtype),
        reliability=torch.empty((0,1), device=device, dtype=dtype),
        status=torch.empty((0,), device=device, dtype=torch.long),
        edge_index=torch.empty((2,0), device=device, dtype=torch.long),
        edge_attr=torch.empty((0,22), device=device, dtype=torch.float32),
        batch_index=torch.empty((0,), device=device, dtype=torch.long),
    )

def current_spatial_forward(model, token_cap=2048):
    old_cap = int(model.query_decoder.cfg.max_spatial_tokens)
    model.query_decoder.cfg.max_spatial_tokens = int(token_cap)
    model.cfg.decoder.max_spatial_tokens = int(token_cap)
    try:
        acq = model.acquisition(b["spacing_um"], b["dref_um"])
        pyramid = model.encoder(b["spatial_inputs"], b["spacing_um"], acq, b.get("spatial_padding_mask"))
        e3 = pyramid.features[3]
        e2 = model.decoder.decode_to_e2(e3, pyramid, acq)
        d1, d0, native_mask_features = model.decoder.decode_from_e2(e2, pyramid, acq)
        dense = model.dense_heads(d0)
        pstate, pscore = model.spatial_proposal_generator(
            d0,e2,b["spatial_inputs"],dense,b["instance_labels"],b["spacing_um"],
            pyramid.spacings_um[2],b["dref_um"],b["instance_ids"],b["instance_batch"],
            b["instance_centroids_um"],b.get("spatial_padding_mask")
        )
        dense = dict(dense)
        dense["proposal_score_logits"] = pscore
        temporal = make_empty_temporal(model, e2.dtype)
        q0 = model.query_builder(
            e2,pyramid.spacings_um[2],b["instance_labels"],b["instance_features"],
            b["instance_ids"],b["instance_batch"],b["instance_centroids_um"],b["dref_um"],temporal,
            memory_ablation="full",return_debug=False,full_attention=False,
            proposal_state=pstate,query_mode="spatial_proposals"
        )
        initial_refs = q0.references_cellscale.clone()
        qf, dec = model.query_decoder(
            q0,[e3,e2,d1],[pyramid.spacings_um[3],pyramid.spacings_um[2],pyramid.spacings_um[1]],
            b["instance_labels"],b["dref_um"],temporal,
            memory_ablation="full",return_debug=False,full_attention=False
        )
        final = dec[-1]
        output = StirNetOutput(
            exist_logits=final["exist_logits"],
            centers_cellscale=final["centers_cellscale"],
            coarse_mask_logits=final["coarse_mask_logits"],
            coarse_spacing_um=final["coarse_spacing_um"],
            query_embeddings=qf.embeddings,
            native_mask_embeddings=model.native_mask_head(qf.embeddings),
            query_types=qf.query_types,
            query_padding_mask=qf.padding_mask,
            source_instance_ids=qf.source_instance_ids,
            query_initial_references_cellscale=initial_refs,
            temporal_salience=qf.temporal_salience,
            temporal_reliability=qf.temporal_reliability,
            aux_outputs=dec[:-1],
            dense_outputs=dense,
            mask_features=native_mask_features,
            spacing_um=b["spacing_um"],
            dref_um=b["dref_um"],
            instance_labels=b["instance_labels"],
            debug=None,
            proposals=pstate,
        )
        inter = {"pyramid":pyramid,"e3":e3,"e2":e2,"d1":d1,"d0":d0,"dense":dense,"proposals":pstate,"q0":q0,"qf":qf,"decoder_outputs":dec}
        return output, inter
    finally:
        model.query_decoder.cfg.max_spatial_tokens = old_cap
        model.cfg.decoder.max_spatial_tokens = old_cap

torch.cuda.reset_peak_memory_stats()
with torch.no_grad(), torch.autocast(device_type="cuda", dtype=AMP_DTYPE):
    baseline_outputs, baseline_intermediate = current_spatial_forward(model, 2048)

log(f"Baseline coarse shape={tuple(baseline_outputs.coarse_mask_logits.shape[-3:])}, peak={torch.cuda.max_memory_allocated()/1024**3:.2f} GiB")


## Experiment A — Initial vs final center failure

In [ ]:
def source9_query_rows(outputs):
    return torch.nonzero(
        (~outputs.query_padding_mask[0].detach().cpu())
        & (outputs.query_types[0].detach().cpu() == QUERY_SPATIAL_PROPOSAL)
        & (outputs.source_instance_ids[0].detach().cpu() == SOURCE_ID),
        as_tuple=False,
    ).flatten()

def center_set_metrics(refs, gt, name):
    refs = refs.detach().float().cpu()
    gt = gt.detach().float().cpu()
    dist = torch.cdist(refs, gt)
    gt_nearest = dist.min(dim=0).values
    q_nearest_gt = dist.argmin(dim=1)
    nearest_counts = torch.bincount(q_nearest_gt, minlength=len(gt))
    r,c = linear_sum_assignment(dist.numpy())
    matched = dist[r,c]
    return {
        "name":name,
        "query_count":len(refs),
        "recall_0p5":float((gt_nearest <= 0.5).float().mean()),
        "recall_1p0":float((gt_nearest <= 1.0).float().mean()),
        "hungarian_mean_dref":float(matched.mean()),
        "hungarian_max_dref":float(matched.max()),
        "hungarian_mean_um":float(matched.mean()*dref_um),
        "duplicate_nearest_gt_count":int((nearest_counts > 1).sum()),
        "missing_gt_0p5":int((gt_nearest > 0.5).sum()),
    }

s9_rows = source9_query_rows(baseline_outputs)
initial_refs = baseline_outputs.query_initial_references_cellscale[0,s9_rows.to(device)].detach().float().cpu()
final_refs = baseline_outputs.centers_cellscale[0,s9_rows.to(device)].detach().float().cpu()

initial_summary = center_set_metrics(initial_refs, source9_gt_centers, "initial")
final_summary = center_set_metrics(final_refs, source9_gt_centers, "final")
movement = torch.linalg.vector_norm(final_refs-initial_refs, dim=-1)

center_audit = pd.DataFrame([initial_summary, final_summary])
display(center_audit)
print(f"Movement mean={movement.mean():.3f} dref, max={movement.max():.3f} dref")

center_audit.to_csv(RUN_DIR/"A_center_audit.csv", index=False)
record_result("A_center_audit", initial=initial_summary, final=final_summary, movement_mean_dref=float(movement.mean()), movement_max_dref=float(movement.max()))


## Experiment B — Proposal decision logic: independent vs relational set reasoning

The current proposal score field has rich spatial evidence, but the final selection rule is local maxima + fixed NMS. This phase deliberately lowers NMS to produce an overcomplete candidate set, freezes STIR-Net, and compares two tiny temporary refiners that receive exactly the same candidate information:

- **Independent refiner:** each candidate processed alone.
- **Set refiner:** candidates self-attend before predicting existence and center offset.

If the set model improves one-to-one coverage or duplicate suppression beyond the independent model, relational proposal reasoning is causally useful.


In [ ]:
class IndependentProposalRefiner(nn.Module):
    def __init__(self, input_dim, hidden=64):
        super().__init__()
        self.body = nn.Sequential(nn.Linear(input_dim,hidden),nn.SiLU(),nn.Linear(hidden,hidden),nn.SiLU())
        self.exist = nn.Linear(hidden,1)
        self.delta = nn.Linear(hidden,3)
    def forward(self,x,refs):
        h = self.body(x)
        return self.exist(h).squeeze(-1), refs + torch.tanh(self.delta(h))*PROPOSAL_MAX_MOVE_DREF

class ProposalSetRefiner(nn.Module):
    def __init__(self, input_dim, d_model=64):
        super().__init__()
        self.input_proj = nn.Linear(input_dim,d_model)
        layer = nn.TransformerEncoderLayer(
            d_model=d_model,nhead=4,dim_feedforward=128,dropout=0.0,
            activation="gelu",batch_first=True,norm_first=True
        )
        self.encoder = nn.TransformerEncoder(layer, num_layers=2)
        self.exist = nn.Linear(d_model,1)
        self.delta = nn.Linear(d_model,3)
    def forward(self,x,refs):
        h = self.encoder(self.input_proj(x)[None])[0]
        return self.exist(h).squeeze(-1), refs + torch.tanh(self.delta(h))*PROPOSAL_MAX_MOVE_DREF

def proposal_refiner_loss(exist_logits, refs, gt):
    dist = torch.cdist(refs.float(), gt.float())
    rr,cc = linear_sum_assignment(dist.detach().cpu().numpy())
    rr = torch.as_tensor(rr,device=device,dtype=torch.long)
    cc = torch.as_tensor(cc,device=device,dtype=torch.long)
    target_exist = torch.zeros_like(exist_logits)
    target_exist[rr] = 1.0
    pos_weight = torch.tensor(max(1.0,(len(exist_logits)-len(rr))/max(len(rr),1)),device=device)
    exist_loss = F.binary_cross_entropy_with_logits(exist_logits,target_exist,pos_weight=pos_weight)
    center_loss = F.smooth_l1_loss(refs[rr],gt[cc],beta=0.10)
    return exist_loss + 4.0*center_loss

def evaluate_refiner(exist_logits, refs, gt, name):
    k = min(len(gt),len(refs))
    keep = torch.topk(exist_logits,k=k).indices
    out = center_set_metrics(refs[keep],gt,name)
    out["selected_count"] = k
    out["mean_selected_exist_prob"] = float(exist_logits[keep].sigmoid().mean().detach().cpu())
    return out

proposal_set_results = []
set_refiner_best_state = None

if RUN_PROPOSAL_SET_EXPERIMENT:
    try:
        old_nms = float(model.spatial_proposal_generator.cfg.nms_radius_dref)
        model.spatial_proposal_generator.cfg.nms_radius_dref = OVERCOMPLETE_NMS_DREF
        with torch.no_grad(), torch.autocast(device_type="cuda",dtype=AMP_DTYPE):
            pstate_over,_ = model.spatial_proposal_generator(
                baseline_intermediate["d0"],baseline_intermediate["e2"],b["spatial_inputs"],
                baseline_intermediate["dense"],b["instance_labels"],b["spacing_um"],
                baseline_intermediate["pyramid"].spacings_um[2],b["dref_um"],
                b["instance_ids"],b["instance_batch"],b["instance_centroids_um"],
                b.get("spatial_padding_mask")
            )
        model.spatial_proposal_generator.cfg.nms_radius_dref = old_nms

        valid = ~pstate_over.padding_mask[0]
        source9 = pstate_over.source_instance_ids[0] == SOURCE_ID
        prows = torch.nonzero(valid & source9, as_tuple=False).flatten()
        if len(prows) < len(source9_gt_ids):
            raise RuntimeError(f"Only {len(prows)} source-9 overcomplete candidates for 9 GT cells.")

        p_emb = pstate_over.embeddings[0,prows].detach().float()
        p_refs = pstate_over.references_cellscale[0,prows].detach().float()
        p_scores = pstate_over.scores[0,prows].detach().float()
        x = torch.cat([p_emb,p_scores[:,None],p_refs],dim=-1)
        gt = source9_gt_centers.to(device).float()

        raw = center_set_metrics(p_refs,gt,"overcomplete_raw")
        raw["selected_count"] = len(p_refs)
        proposal_set_results.append(raw)

        for model_name,cls in [("independent_refiner",IndependentProposalRefiner),("set_refiner",ProposalSetRefiner)]:
            torch.manual_seed(SEED)
            refiner = cls(x.shape[-1]).to(device)
            opt = torch.optim.AdamW(refiner.parameters(),lr=PROPOSAL_REFINER_LR,weight_decay=1e-4)
            best = None
            for step in range(PROPOSAL_REFINER_STEPS+1):
                refiner.train()
                opt.zero_grad(set_to_none=True)
                exist,refined = refiner(x,p_refs)
                loss = proposal_refiner_loss(exist,refined,gt)
                if step < PROPOSAL_REFINER_STEPS:
                    loss.backward()
                    torch.nn.utils.clip_grad_norm_(refiner.parameters(),1.0)
                    opt.step()
                if step==0 or step%50==0 or step==PROPOSAL_REFINER_STEPS:
                    refiner.eval()
                    with torch.no_grad():
                        e2,r2 = refiner(x,p_refs)
                        s = evaluate_refiner(e2,r2,gt,f"{model_name}_step_{step}")
                        s["step"] = step
                        s["loss"] = float(loss.detach().cpu())
                    log(f"B {model_name} step={step} recall0.5={s['recall_0p5']:.3f} mean={s['hungarian_mean_dref']:.3f} dup={s['duplicate_nearest_gt_count']}")
                    rank = (s["recall_0p5"],-s["hungarian_mean_dref"],-s["duplicate_nearest_gt_count"])
                    if best is None or rank > best["rank"]:
                        best = {"rank":rank,"summary":dict(s),"state":copy.deepcopy(refiner.state_dict())}

            clean = dict(best["summary"])
            clean["name"] = model_name
            proposal_set_results.append(clean)

            if model_name == "set_refiner":
                set_refiner_best_state = best["state"]
                set_refiner_x = x.detach()
                set_refiner_refs0 = p_refs.detach()

            del refiner,opt
            cleanup()

        proposal_set_df = pd.DataFrame(proposal_set_results)
        display(proposal_set_df)
        proposal_set_df.to_csv(RUN_DIR/"B_proposal_set_reasoning.csv",index=False)
        record_result("B_proposal_set_reasoning",rows=proposal_set_df.to_dict("records"))
    except Exception as exc:
        record_error("B_proposal_set_reasoning",exc)
        proposal_set_df = pd.DataFrame(proposal_set_results)
        cleanup()
else:
    proposal_set_df = pd.DataFrame()


## Shared high-resolution local-evidence sampler

The next two experiments use the exact native/high-resolution evidence available upstream, but avoid compressing it to center/mean/max statistics before making the decision.

Evidence tensor:

```text
D0
+ five current spatial inputs
+ dense foreground / center / boundary probabilities
+ proposal-relative xyz
```


In [ ]:
def physical_grid(ref_cellscale, shape, spacing_um, dref, grid_size=LOCAL_GRID_SIZE, extent_dref=LOCAL_EXTENT_DREF):
    axis = torch.linspace(-extent_dref,extent_dref,grid_size,device=device,dtype=torch.float32)*dref.float()
    offsets = torch.stack(torch.meshgrid(axis,axis,axis,indexing="ij"),dim=-1)
    center_um = ref_cellscale.float()*dref.float()
    points = center_um[None,None,None,:] + offsets
    extent = torch.tensor([shape[0]-1,shape[1]-1,shape[2]-1],device=device,dtype=torch.float32)*spacing_um.float()
    norm_zyx = points/(0.5*extent).clamp_min(1e-8)
    return norm_zyx[..., [2,1,0]]

@torch.no_grad()
def sample_feature_cubes(feature, refs, spacing_um, dref):
    cubes = []
    for ref in refs:
        grid = physical_grid(ref,tuple(int(v) for v in feature.shape[-3:]),spacing_um,dref)[None]
        with torch.autocast(device_type="cuda",dtype=AMP_DTYPE):
            sampled = F.grid_sample(feature,grid,mode="bilinear",padding_mode="zeros",align_corners=True)[0]
        cubes.append(sampled.float())
    return torch.stack(cubes,dim=0)

def relative_xyz(count):
    axis = torch.linspace(-LOCAL_EXTENT_DREF,LOCAL_EXTENT_DREF,LOCAL_GRID_SIZE,device=device)
    zz,yy,xx = torch.meshgrid(axis,axis,axis,indexing="ij")
    rel = torch.stack([zz,yy,xx],dim=0)
    return rel[None].expand(count,-1,-1,-1,-1)

@torch.no_grad()
def explicit_evidence(refs):
    d0 = sample_feature_cubes(
        baseline_intermediate["d0"],refs,b["spacing_um"][0],b["dref_um"][0]
    )
    inputs = sample_feature_cubes(
        b["spatial_inputs"],refs,b["spacing_um"][0],b["dref_um"][0]
    )
    dense_full = torch.cat([
        baseline_intermediate["dense"]["foreground_logits"].sigmoid(),
        baseline_intermediate["dense"]["center_heatmap_logits"].sigmoid(),
        baseline_intermediate["dense"]["boundary_logits"].sigmoid(),
    ],dim=1)
    dense = sample_feature_cubes(
        dense_full,refs,b["spacing_um"][0],b["dref_um"][0]
    )
    return {"d0":d0,"inputs":inputs,"dense":dense,"rel":relative_xyz(len(refs))}


## Experiment C — Center information sufficiency

The current center head only sees the compressed query vector. This temporary refiner receives the direct local 3-D tensor **plus** the same query representation. Only this tiny head is trained.


In [ ]:
class LocalCenterRefiner(nn.Module):
    def __init__(self,in_ch,qdim):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv3d(in_ch,24,3,padding=1),nn.GroupNorm(4,24),nn.SiLU(),
            nn.Conv3d(24,32,3,padding=1),nn.GroupNorm(4,32),nn.SiLU(),
            nn.AdaptiveAvgPool3d(1)
        )
        self.q = nn.Sequential(nn.Linear(qdim,32),nn.SiLU())
        self.out = nn.Sequential(nn.Linear(64,64),nn.SiLU(),nn.Linear(64,3))
    def forward(self,evidence,q):
        s = self.conv(evidence).flatten(1)
        qh = self.q(q)
        return torch.tanh(self.out(torch.cat([s,qh],dim=-1)))*0.75

local_center_result = None
local_center_refs = None

if RUN_LOCAL_CENTER_EXPERIMENT:
    try:
        dist = torch.cdist(initial_refs,source9_gt_centers)
        rr,cc = linear_sum_assignment(dist.numpy())
        rr = torch.as_tensor(rr,dtype=torch.long)
        cc = torch.as_tensor(cc,dtype=torch.long)

        anchors = initial_refs[rr].to(device)
        matched_gt = source9_gt_centers[cc].to(device)
        qemb = baseline_intermediate["q0"].embeddings[0,s9_rows[rr].to(device)].detach().float()

        ev = explicit_evidence(anchors)
        all_ev = torch.cat([ev["d0"],ev["inputs"],ev["dense"],ev["rel"]],dim=1)

        torch.manual_seed(SEED)
        refiner = LocalCenterRefiner(all_ev.shape[1],qemb.shape[-1]).to(device)
        opt = torch.optim.AdamW(refiner.parameters(),lr=LOCAL_HEAD_LR,weight_decay=1e-4)
        target_delta = matched_gt-anchors

        for step in range(LOCAL_CENTER_STEPS+1):
            refiner.train()
            opt.zero_grad(set_to_none=True)
            delta = refiner(all_ev,qemb)
            loss = F.smooth_l1_loss(delta,target_delta,beta=0.05)
            if step < LOCAL_CENTER_STEPS:
                loss.backward()
                torch.nn.utils.clip_grad_norm_(refiner.parameters(),1.0)
                opt.step()
            if step==0 or step%50==0 or step==LOCAL_CENTER_STEPS:
                with torch.no_grad():
                    r = anchors+refiner(all_ev,qemb)
                    err = torch.linalg.vector_norm(r-matched_gt,dim=-1)
                log(f"C local tensor step={step} mean={err.mean():.3f} dref max={err.max():.3f}")

        with torch.no_grad():
            local_center_refs = (anchors+refiner(all_ev,qemb)).detach()

        current_final = final_refs[rr].to(device)
        initial_err = torch.linalg.vector_norm(anchors-matched_gt,dim=-1)
        final_err = torch.linalg.vector_norm(current_final-matched_gt,dim=-1)
        local_err = torch.linalg.vector_norm(local_center_refs-matched_gt,dim=-1)

        local_center_result = {
            "initial_mean_dref":float(initial_err.mean()),
            "current_final_mean_dref":float(final_err.mean()),
            "local_tensor_mean_dref":float(local_err.mean()),
            "initial_max_dref":float(initial_err.max()),
            "current_final_max_dref":float(final_err.max()),
            "local_tensor_max_dref":float(local_err.max()),
        }
        display(pd.DataFrame([local_center_result]))

        pd.DataFrame({
            "gt_id":source9_gt_ids[cc.numpy()],
            "initial_error_dref":initial_err.cpu().numpy(),
            "current_final_error_dref":final_err.cpu().numpy(),
            "local_tensor_error_dref":local_err.cpu().numpy(),
        }).to_csv(RUN_DIR/"C_local_center_per_cell.csv",index=False)

        local_center_match_gt_rows = cc
        record_result("C_local_center_information",**local_center_result)
        del refiner,opt
        cleanup()
    except Exception as exc:
        record_error("C_local_center_information",exc)
        cleanup()


## Experiment D — Coarse mask target erasure vs mask-lattice resolution

The current final coarse mask is a query-derived mask vector dotted with a token-capped spatial feature map. This phase keeps that **same dot-product mask design** and trains only copied final mask heads.

Variants:

- `2048 + nearest`: current target construction failure mode.
- `2048 + occupancy`: same lattice, but each GT instance is downsampled independently with max pooling.
- `8192 + occupancy`
- `16384 + occupancy`

If occupancy removes zero-voxel GT cells, target construction is proven defective.  
If larger lattices improve Dice with the same dot-product mechanism, attention token compression and mask resolution should be decoupled.


In [ ]:
def cap_feature_tokens_local(feature,spacing_um,max_tokens):
    z,y,x = feature.shape[-3:]
    n = z*y*x
    if n <= max_tokens:
        return feature,spacing_um
    scale = (n/max_tokens)**(1/3)
    target_shape = tuple(max(1,int(round(v/scale))) for v in (z,y,x))
    while np.prod(target_shape) > max_tokens:
        k = max(range(3),key=lambda i:target_shape[i])
        target_shape = tuple(v-1 if i==k and v>1 else v for i,v in enumerate(target_shape))
    pooled = F.adaptive_avg_pool3d(feature,target_shape)
    ratio = torch.tensor([
        (size-1)/(out-1) if out>1 else 1.0
        for size,out in zip((z,y,x),target_shape)
    ],device=feature.device,dtype=spacing_um.dtype)
    return pooled,spacing_um*ratio[None]

def occupancy_targets(shape):
    masks = []
    for gt_id in source9_gt_ids:
        native = torch.from_numpy((gt_labels_native==int(gt_id)).astype(np.float32))[None,None]
        pooled = F.adaptive_max_pool3d(native,shape)[0,0]
        masks.append(pooled)
    return torch.stack(masks).to(device)

def nearest_targets(shape):
    return target_masks_at_shape(
        target,shape,device,target_indices=source9_gt_indices
    )

def fixed_assignment(query_centers):
    dist = torch.cdist(query_centers.detach().float().cpu(),source9_gt_centers.float())
    rr,cc = linear_sum_assignment(dist.numpy())
    return torch.as_tensor(rr,dtype=torch.long),torch.as_tensor(cc,dtype=torch.long)

def mask_metrics(logits,gt,support):
    p = logits.float().sigmoid()
    plocal = p*support.float()
    inter = (plocal*gt).flatten(1).sum(-1)
    soft = (2*inter+1e-6)/(plocal.flatten(1).sum(-1)+gt.flatten(1).sum(-1)+1e-6)
    hard = (p>=0.5)&support.bool()
    gb = gt.bool()
    tp = (hard&gb).flatten(1).sum(-1).float()
    fp = (hard&~gb).flatten(1).sum(-1).float()
    fn = (~hard&gb).flatten(1).sum(-1).float()
    hd = (2*tp+1e-6)/(2*tp+fp+fn+1e-6)
    return soft,hd

class DotMaskProbe(nn.Module):
    def __init__(self,mask_embed,mask_proj):
        super().__init__()
        self.mask_embed = copy.deepcopy(mask_embed)
        self.mask_proj = copy.deepcopy(mask_proj)
    def forward(self,q,spatial):
        emb = self.mask_embed(q)
        feat = self.mask_proj(spatial)[0]
        return torch.einsum("qc,cv->qv",emb,feat.flatten(1)).reshape(len(q),*feat.shape[-3:])

def train_dot_probe(token_cap,target_kind):
    d1 = baseline_intermediate["d1"].detach()
    spacing_d1 = baseline_intermediate["pyramid"].spacings_um[1]
    d1cap,spacing_cap = cap_feature_tokens_local(d1,spacing_d1,token_cap)
    shape = tuple(int(v) for v in d1cap.shape[-3:])

    gt_all = nearest_targets(shape) if target_kind=="nearest" else occupancy_targets(shape)
    rr,cc = fixed_assignment(final_refs)
    qrows = s9_rows[rr].to(device)
    q = baseline_outputs.query_embeddings[0,qrows].detach().float()
    gt = gt_all[cc.to(device)].float()
    centers = source9_gt_centers[cc].to(device)
    support = build_local_support_masks(
        gt,centers,spacing_cap[0],b["dref_um"][0],float(cfg.losses.mask_supervision_radius_dref)
    )

    probe = DotMaskProbe(
        model.query_decoder.layers[-1].mask_embed,
        model.query_decoder.mask_feature_proj[-1],
    ).to(device).float()
    opt = torch.optim.AdamW(probe.parameters(),lr=MASK_PROBE_LR,weight_decay=1e-4)
    spatial = d1cap.detach().float()

    for step in range(MASK_PROBE_STEPS+1):
        probe.train()
        opt.zero_grad(set_to_none=True)
        logits = probe(q,spatial)
        p = logits.sigmoid()
        plocal = p*support.float()
        inter = (plocal*gt).flatten(1).sum(-1)
        dice_loss = 1-(2*inter+1e-6)/(plocal.flatten(1).sum(-1)+gt.flatten(1).sum(-1)+1e-6)
        bce = torch.stack([
            F.binary_cross_entropy_with_logits(logits[i][support[i]],gt[i][support[i]])
            for i in range(len(gt))
        ]).mean()
        loss = dice_loss.mean()+0.5*bce
        if step < MASK_PROBE_STEPS:
            loss.backward()
            torch.nn.utils.clip_grad_norm_(probe.parameters(),1.0)
            opt.step()

    probe.eval()
    with torch.no_grad():
        logits = probe(q,spatial)
        soft,hard = mask_metrics(logits,gt,support)

    counts = gt.flatten(1).sum(-1).detach().cpu().numpy()
    result = {
        "token_cap":token_cap,
        "target_kind":target_kind,
        "shape":str(shape),
        "spatial_positions":int(np.prod(shape)),
        "zero_gt_cells":int((counts==0).sum()),
        "mean_gt_positive_voxels":float(counts.mean()),
        "min_gt_positive_voxels":float(counts.min()),
        "soft_dice_mean":float(soft.mean().cpu()),
        "soft_dice_min":float(soft.min().cpu()),
        "hard_dice_mean":float(hard.mean().cpu()),
    }
    detail = pd.DataFrame({
        "gt_id":source9_gt_ids[cc.numpy()],
        "positive_target_voxels":counts,
        "soft_dice":soft.cpu().numpy(),
        "hard_dice":hard.cpu().numpy(),
    })
    del probe,opt,logits,gt_all
    cleanup()
    return result,detail

coarse_probe_results = []

if RUN_COARSE_MASK_PROBE:
    try:
        for cap,target_kind in [
            (2048,"nearest"),
            (2048,"occupancy"),
            (8192,"occupancy"),
            (16384,"occupancy"),
        ]:
            log(f"D probe cap={cap} target={target_kind}")
            result,detail = train_dot_probe(cap,target_kind)
            coarse_probe_results.append(result)
            detail.to_csv(RUN_DIR/f"D_{cap}_{target_kind}_per_cell.csv",index=False)
            record_result("D_coarse_dot_mask_probe",**result)
            log(f"D result cap={cap} target={target_kind} Dice={result['soft_dice_mean']:.3f} zeroGT={result['zero_gt_cells']}")

        coarse_probe_df = pd.DataFrame(coarse_probe_results)
        display(coarse_probe_df)
        coarse_probe_df.to_csv(RUN_DIR/"D_coarse_dot_mask_probe.csv",index=False)
    except Exception as exc:
        record_error("D_coarse_dot_mask_probe",exc)
        coarse_probe_df = pd.DataFrame(coarse_probe_results)
        cleanup()
else:
    coarse_probe_df = pd.DataFrame()


## Experiment E — Explicit local evidence for instance masks

The current mask decision receives a compressed query embedding and learned mask feature. This temporary local head gets progressively richer information:

1. `D0 + relative xyz + query`
2. `D0 + raw + relative xyz + query`
3. `D0 + all five inputs + dense predictions + relative xyz + query`

The strongest variant is then repeated using GT centers instead of predicted centers.

This directly tests whether raw/EDT/boundary/mask/marker evidence should remain explicit at the instance-mask decision.


In [ ]:
class LocalEvidenceMaskHead(nn.Module):
    def __init__(self,in_ch,qdim,hidden=32):
        super().__init__()
        self.evidence = nn.Sequential(
            nn.Conv3d(in_ch,hidden,3,padding=1),nn.GroupNorm(4,hidden),nn.SiLU(),
            nn.Conv3d(hidden,hidden,3,padding=1),nn.GroupNorm(4,hidden),nn.SiLU()
        )
        self.q = nn.Sequential(nn.Linear(qdim,hidden),nn.SiLU(),nn.Linear(hidden,hidden))
        self.fuse = nn.Sequential(nn.Conv3d(2*hidden,hidden,1),nn.SiLU(),nn.Conv3d(hidden,1,1))
    def forward(self,evidence,q):
        f = self.evidence(evidence)
        qh = self.q(q)[:,:,None,None,None].expand_as(f)
        return self.fuse(torch.cat([f,qh],dim=1))[:,0]

@torch.no_grad()
def sample_gt_cubes(gt_ids_ordered,refs):
    out = []
    for gt_id,ref in zip(gt_ids_ordered,refs):
        native = torch.from_numpy((gt_labels_native==int(gt_id)).astype(np.float32))[None,None].to(device)
        grid = physical_grid(ref,tuple(int(v) for v in native.shape[-3:]),b["spacing_um"][0],b["dref_um"][0])[None]
        sampled = F.grid_sample(native,grid,mode="nearest",padding_mode="zeros",align_corners=True)[0,0]
        out.append(sampled)
        del native
        cleanup()
    return torch.stack(out,dim=0)

def train_local_mask(name,refs,gt_ids_ordered,qemb,variant):
    parts = explicit_evidence(refs)
    if variant=="d0":
        evidence = torch.cat([parts["d0"],parts["rel"]],dim=1)
    elif variant=="d0_raw":
        evidence = torch.cat([parts["d0"],parts["inputs"][:,0:1],parts["rel"]],dim=1)
    elif variant=="explicit_all":
        evidence = torch.cat([parts["d0"],parts["inputs"],parts["dense"],parts["rel"]],dim=1)
    else:
        raise ValueError(variant)

    gt = sample_gt_cubes(gt_ids_ordered,refs).float()
    torch.manual_seed(SEED)
    head = LocalEvidenceMaskHead(evidence.shape[1],qemb.shape[-1]).to(device)
    opt = torch.optim.AdamW(head.parameters(),lr=LOCAL_HEAD_LR,weight_decay=1e-4)
    best = (-1.0,None)

    for step in range(LOCAL_MASK_STEPS+1):
        head.train()
        opt.zero_grad(set_to_none=True)
        logits = head(evidence,qemb)
        p = logits.sigmoid()
        inter = (p*gt).flatten(1).sum(-1)
        dice = (2*inter+1e-6)/(p.flatten(1).sum(-1)+gt.flatten(1).sum(-1)+1e-6)
        loss = (1-dice.mean())+0.5*F.binary_cross_entropy_with_logits(logits,gt)
        if step < LOCAL_MASK_STEPS:
            loss.backward()
            torch.nn.utils.clip_grad_norm_(head.parameters(),1.0)
            opt.step()
        if step==0 or step%30==0 or step==LOCAL_MASK_STEPS:
            mdice = float(dice.mean().detach().cpu())
            log(f"E {name} step={step} Dice={mdice:.3f}")
            if mdice > best[0]:
                best = (mdice,copy.deepcopy(head.state_dict()))

    head.load_state_dict(best[1])
    head.eval()
    with torch.no_grad():
        p = head(evidence,qemb).sigmoid()
        inter = (p*gt).flatten(1).sum(-1)
        soft = (2*inter+1e-6)/(p.flatten(1).sum(-1)+gt.flatten(1).sum(-1)+1e-6)
        hard = p>=0.5
        gb = gt.bool()
        tp = (hard&gb).flatten(1).sum(-1).float()
        fp = (hard&~gb).flatten(1).sum(-1).float()
        fn = (~hard&gb).flatten(1).sum(-1).float()
        hd = (2*tp+1e-6)/(2*tp+fp+fn+1e-6)

    result = {
        "name":name,
        "variant":variant,
        "soft_dice_mean":float(soft.mean().cpu()),
        "soft_dice_min":float(soft.min().cpu()),
        "hard_dice_mean":float(hd.mean().cpu()),
        "evidence_channels":int(evidence.shape[1]),
    }
    detail = pd.DataFrame({"gt_id":gt_ids_ordered,"soft_dice":soft.cpu().numpy(),"hard_dice":hd.cpu().numpy()})
    pred = p.detach().cpu().to(torch.float16).numpy()
    del head,opt,evidence,gt
    cleanup()
    return result,detail,pred

local_mask_results = []
local_mask_predictions = {}

if RUN_LOCAL_MASK_EXPERIMENT:
    try:
        rr,cc = fixed_assignment(final_refs)
        qrows = s9_rows[rr].to(device)
        predicted_refs = final_refs[rr].to(device)
        ordered_gt_ids = source9_gt_ids[cc.numpy()]
        oracle_refs = source9_gt_centers[cc].to(device)
        qemb = baseline_outputs.query_embeddings[0,qrows].detach().float()

        for variant in ["d0","d0_raw","explicit_all"]:
            name = f"predicted_center_{variant}"
            result,detail,pred = train_local_mask(name,predicted_refs,ordered_gt_ids,qemb,variant)
            local_mask_results.append(result)
            local_mask_predictions[name] = pred
            detail.to_csv(RUN_DIR/f"E_{name}_per_cell.csv",index=False)
            record_result("E_local_mask_information",**result)

        result,detail,pred = train_local_mask(
            "oracle_center_explicit_all",oracle_refs,ordered_gt_ids,qemb,"explicit_all"
        )
        local_mask_results.append(result)
        local_mask_predictions[result["name"]] = pred
        detail.to_csv(RUN_DIR/"E_oracle_center_explicit_all_per_cell.csv",index=False)
        record_result("E_local_mask_information",**result)

        local_mask_df = pd.DataFrame(local_mask_results)
        display(local_mask_df)
        local_mask_df.to_csv(RUN_DIR/"E_local_mask_information.csv",index=False)

        np.savez_compressed(
            RUN_DIR/"E_local_mask_predictions_small.npz",
            gt_ids=ordered_gt_ids,
            predicted_centers=predicted_refs.detach().cpu().numpy(),
            gt_centers=oracle_refs.detach().cpu().numpy(),
            **local_mask_predictions,
        )
    except Exception as exc:
        record_error("E_local_mask_information",exc)
        local_mask_df = pd.DataFrame(local_mask_results)
        cleanup()
else:
    local_mask_df = pd.DataFrame()


## Experiment F — Support-radius geometry

Measure how much of each real source-9 cell is geometrically covered by radial supports around GT centers and current predicted centers.

This does not depend on the confounded Notebook-27 native checkpoint.


In [ ]:
support_rows = []

if RUN_SUPPORT_AUDIT:
    try:
        rr,cc = fixed_assignment(final_refs)
        pred_centers = final_refs[rr].numpy()
        gt_centers = source9_gt_centers[cc].numpy()
        gt_ids_ordered = source9_gt_ids[cc.numpy()]

        full_shape = np.asarray(gt_labels_native.shape,dtype=np.float64)
        extent_um = (full_shape-1)*spacing_native
        voxel_volume = float(np.prod(spacing_native))

        for gt_id,gc,pc in zip(gt_ids_ordered,gt_centers,pred_centers):
            vox = np.argwhere(gt_labels_native==int(gt_id)).astype(np.float64)
            vox_um = vox*spacing_native[None]-0.5*extent_um[None]
            dg = np.linalg.norm(vox_um-(gc*dref_um)[None],axis=1)
            dp = np.linalg.norm(vox_um-(pc*dref_um)[None],axis=1)
            for rd in SUPPORT_RADII_DREF:
                ru = rd*dref_um
                sphere_vox = (4/3)*math.pi*ru**3/voxel_volume
                support_rows.append({
                    "gt_id":int(gt_id),
                    "radius_dref":rd,
                    "coverage_from_gt_center":float((dg<=ru).mean()),
                    "coverage_from_pred_center":float((dp<=ru).mean()),
                    "sphere_to_gt_volume_ratio":float(sphere_vox/max(len(vox),1)),
                    "gt_voxels":len(vox),
                })

        support_df = pd.DataFrame(support_rows)
        support_summary = support_df.groupby("radius_dref").agg(
            gt_center_coverage_mean=("coverage_from_gt_center","mean"),
            gt_center_coverage_min=("coverage_from_gt_center","min"),
            pred_center_coverage_mean=("coverage_from_pred_center","mean"),
            pred_center_coverage_min=("coverage_from_pred_center","min"),
            sphere_to_gt_volume_ratio_mean=("sphere_to_gt_volume_ratio","mean"),
        ).reset_index()

        display(support_summary)
        support_df.to_csv(RUN_DIR/"F_support_per_cell.csv",index=False)
        support_summary.to_csv(RUN_DIR/"F_support_summary.csv",index=False)
        record_result("F_support_geometry",rows=support_summary.to_dict("records"))
    except Exception as exc:
        record_error("F_support_geometry",exc)
        support_df = pd.DataFrame()
        support_summary = pd.DataFrame()
        cleanup()
else:
    support_df = pd.DataFrame()
    support_summary = pd.DataFrame()


## 7. Cross-experiment causal diagnosis

This cell converts the measurements into architecture implications. It does not assume every proposed patch is good; a patch is recommended only if its causal experiment supports it.


In [ ]:
diagnosis = {
    "generated_at": now_text(),
    "checkpoint": str(BEST_SPATIAL_QUERY_CHECKPOINT),
    "source9_gt_count": len(source9_gt_ids),
    "observations": [],
    "architecture_implications": [],
}

diagnosis["observations"].append({
    "topic":"center_initial_vs_final",
    "initial_recall_0p5":initial_summary["recall_0p5"],
    "final_recall_0p5":final_summary["recall_0p5"],
    "initial_mean_dref":initial_summary["hungarian_mean_dref"],
    "final_mean_dref":final_summary["hungarian_mean_dref"],
})

if abs(initial_summary["hungarian_mean_dref"]-final_summary["hungarian_mean_dref"]) < 0.10:
    diagnosis["architecture_implications"].append(
        "Initial and final center quality are similar; the center error begins at proposal localization and is not mainly created by later query refinement."
    )

if len(proposal_set_df):
    independent = proposal_set_df[proposal_set_df["name"]=="independent_refiner"]
    setrow = proposal_set_df[proposal_set_df["name"]=="set_refiner"]
    if len(independent) and len(setrow):
        i = independent.iloc[0]
        s = setrow.iloc[0]
        set_better = (
            s["recall_0p5"] > i["recall_0p5"] + 0.05
            or (
                s["duplicate_nearest_gt_count"] < i["duplicate_nearest_gt_count"]
                and s["hungarian_mean_dref"] < i["hungarian_mean_dref"]
            )
        )
        diagnosis["observations"].append({
            "topic":"proposal_relational_reasoning",
            "independent_recall_0p5":float(i["recall_0p5"]),
            "set_recall_0p5":float(s["recall_0p5"]),
            "independent_mean_dref":float(i["hungarian_mean_dref"]),
            "set_mean_dref":float(s["hungarian_mean_dref"]),
            "set_better":bool(set_better),
        })
        diagnosis["architecture_implications"].append(
            "Proposal-set reasoning improved duplicate/coverage behavior; add learned candidate-to-candidate refinement."
            if set_better
            else
            "Proposal-set reasoning did not clearly beat an independent refiner here; do not add proposal self-attention solely for deduplication yet."
        )

if local_center_result is not None:
    current_error = local_center_result["current_final_mean_dref"]
    local_error = local_center_result["local_tensor_mean_dref"]
    rel_gain = (current_error-local_error)/max(current_error,1e-8)
    diagnosis["observations"].append({
        "topic":"center_information_sufficiency",
        "current_mean_dref":current_error,
        "local_tensor_mean_dref":local_error,
        "relative_improvement":rel_gain,
    })
    if rel_gain >= 0.20:
        diagnosis["architecture_implications"].append(
            "Direct local 3-D evidence materially improves center localization; center refinement should consume local spatial tensors rather than only the compressed query vector."
        )

if len(coarse_probe_df):
    n2048 = coarse_probe_df[(coarse_probe_df["token_cap"]==2048)&(coarse_probe_df["target_kind"]=="nearest")]
    o2048 = coarse_probe_df[(coarse_probe_df["token_cap"]==2048)&(coarse_probe_df["target_kind"]=="occupancy")]
    occ = coarse_probe_df[coarse_probe_df["target_kind"]=="occupancy"].sort_values("soft_dice_mean",ascending=False)

    if len(n2048) and len(o2048):
        n = n2048.iloc[0]
        o = o2048.iloc[0]
        diagnosis["observations"].append({
            "topic":"coarse_target_construction",
            "nearest_zero_gt_cells":int(n["zero_gt_cells"]),
            "occupancy_zero_gt_cells":int(o["zero_gt_cells"]),
            "nearest_soft_dice":float(n["soft_dice_mean"]),
            "occupancy_soft_dice":float(o["soft_dice_mean"]),
        })
        if n["zero_gt_cells"] > 0 and o["zero_gt_cells"] == 0:
            diagnosis["architecture_implications"].append(
                "Nearest-neighbor resizing of the multiclass label map erases real instances; replace it with per-instance occupancy-preserving target downsampling."
            )

    if len(occ):
        best = occ.iloc[0]
        diagnosis["observations"].append({
            "topic":"coarse_mask_resolution",
            "best_token_cap":int(best["token_cap"]),
            "best_soft_dice":float(best["soft_dice_mean"]),
        })
        if int(best["token_cap"]) > 2048:
            diagnosis["architecture_implications"].append(
                "The same dot-product mask head improves on a denser mask lattice; decouple transformer attention token compression from mask output resolution."
            )

if len(local_mask_df):
    by = local_mask_df.set_index("name")
    needed = ["predicted_center_d0","predicted_center_d0_raw","predicted_center_explicit_all","oracle_center_explicit_all"]
    if all(x in by.index for x in needed):
        d0 = float(by.loc["predicted_center_d0","soft_dice_mean"])
        raw = float(by.loc["predicted_center_d0_raw","soft_dice_mean"])
        allinfo = float(by.loc["predicted_center_explicit_all","soft_dice_mean"])
        oracle = float(by.loc["oracle_center_explicit_all","soft_dice_mean"])
        diagnosis["observations"].append({
            "topic":"mask_information_sufficiency",
            "d0_dice":d0,
            "d0_plus_raw_dice":raw,
            "explicit_all_dice":allinfo,
            "oracle_center_explicit_all_dice":oracle,
        })
        if raw > d0 + 0.03:
            diagnosis["architecture_implications"].append(
                "Direct raw intensity improves instance masks beyond D0 alone; preserve raw evidence explicitly at mask decoding."
            )
        if allinfo > max(d0,raw) + 0.03:
            diagnosis["architecture_implications"].append(
                "Explicit mask/EDT/boundary/marker/dense evidence improves masks; rich local geometry is being lost by the current compressed mask interface."
            )
        if oracle > allinfo + 0.05:
            diagnosis["architecture_implications"].append(
                "Oracle centers improve the same local mask decoder; center accuracy is a direct downstream segmentation bottleneck."
            )
        if allinfo > 0.50:
            diagnosis["architecture_implications"].append(
                "A small anchor-local high-resolution mask decoder can fit source 9 much better than the current coarse pathway; an anchor-conditioned local mask architecture is strongly supported."
            )

if len(support_summary):
    viable = support_summary[support_summary["gt_center_coverage_min"]>=0.99]
    if len(viable):
        smallest = viable.sort_values("radius_dref").iloc[0]
        diagnosis["observations"].append({
            "topic":"support_radius",
            "smallest_gt_99pct_radius_dref":float(smallest["radius_dref"]),
            "pred_center_coverage_min":float(smallest["pred_center_coverage_min"]),
            "sphere_to_gt_volume_ratio_mean":float(smallest["sphere_to_gt_volume_ratio_mean"]),
        })
        if float(smallest["radius_dref"]) < 2.5:
            diagnosis["architecture_implications"].append(
                "A 2.5 dref native support is larger than necessary for GT-centered cells; align render support with the smallest radius that preserves real-cell coverage."
            )

(RUN_DIR/"diagnosis.json").write_text(json.dumps(diagnosis,indent=2),encoding="utf-8")

print("="*88)
print("NOTEBOOK 28 CAUSAL DIAGNOSIS")
print("="*88)
for item in diagnosis["architecture_implications"]:
    print("•",item)
print()
print("Saved:",RUN_DIR/"diagnosis.json")


## 8. Final comparison tables


In [ ]:
print("A — centers")
display(center_audit)

if len(proposal_set_df):
    print("B — proposal set reasoning")
    display(proposal_set_df)

if local_center_result is not None:
    print("C — center information")
    display(pd.DataFrame([local_center_result]))

if len(coarse_probe_df):
    print("D — coarse masks")
    display(coarse_probe_df)

if len(local_mask_df):
    print("E — local mask information")
    display(local_mask_df)

if len(support_summary):
    print("F — support geometry")
    display(support_summary)

print()
print("Run directory:",RUN_DIR)


## 9. Optional Napari center overlay

Set `OPEN_NAPARI_AT_END=True` near the top before running if you want this viewer. It overlays GT, current initial/final, and temporary local-tensor refined centers.


In [ ]:
if OPEN_NAPARI_AT_END:
    import napari

    source_vox = np.argwhere(current_labels_native==SOURCE_ID)
    lo = source_vox.min(axis=0)
    hi = source_vox.max(axis=0)+1
    margin_vox = np.ceil((2.0*dref_um)/spacing_native).astype(int)
    lo = np.maximum(0,lo-margin_vox)
    hi = np.minimum(np.asarray(current_labels_native.shape),hi+margin_vox)
    crop = tuple(slice(int(a),int(z)) for a,z in zip(lo,hi))

    raw_native = batch_cpu["spatial_inputs"][0,0].detach().cpu().float().numpy()
    raw_crop = raw_native[crop]
    full_shape = np.asarray(current_labels_native.shape,dtype=np.float64)
    extent_um = (full_shape-1)*spacing_native

    def refs_to_crop_vox(refs):
        refs = np.asarray(refs)
        vox = (refs*dref_um+0.5*extent_um[None])/spacing_native[None]
        return vox-lo[None]

    viewer = napari.Viewer(title="STIR-Net Notebook 28 — Center Audit")
    scale = tuple(float(v) for v in spacing_native)
    viewer.add_image(raw_crop,name="Raw",scale=scale)
    viewer.add_points(refs_to_crop_vox(source9_gt_centers.numpy()),name="GT centers",scale=scale,size=6)
    viewer.add_points(refs_to_crop_vox(initial_refs.numpy()),name="Initial query centers",scale=scale,size=6)
    viewer.add_points(refs_to_crop_vox(final_refs.numpy()),name="Final query centers",scale=scale,size=6)
    if local_center_refs is not None:
        viewer.add_points(refs_to_crop_vox(local_center_refs.detach().cpu().numpy()),name="Local tensor refined centers",scale=scale,size=7)
    viewer.dims.ndisplay = 3


### Output files

Notebook 28 stores only small experiment outputs:

```text
A_center_audit.csv
B_proposal_set_reasoning.csv
C_local_center_per_cell.csv
D_coarse_dot_mask_probe.csv
E_local_mask_information.csv
E_local_mask_predictions_small.npz
F_support_summary.csv
diagnosis.json
experiment.log
results.jsonl
errors.jsonl   # only if a phase fails
```

No full STIR-Net checkpoint or large native prediction volume is written.
